# Repairing a broken ligand network

In a relative binding free energy (RBFE) campaign some transformations may fail. If
enough edges drop out, the completed network can split into disconnected pieces,
and a disconnected network can no longer rank all of its ligands.

This notebook takes the TYK2 system from the JACS benchmark
set and repairs the network:

1. load the stored, connected **planned** network,
2. parse a directory of **result JSONs** to find which edges actually completed,
3. reconstruct the **completed** network and diff it against the plan,
4. **repair** the breaks in two different ways:
   - by ligand name
   - by lomap score
6. **rebuild** alchemical transformations for the new edges and write them out.

## 1. Fetch the data

In [1]:
import urllib.request

NETWORK_URL = (
    "https://raw.githubusercontent.com/OpenFreeEnergy/openfe-benchmarks/main/"
    "openfe_benchmarks/data/benchmark_systems/jacs_set/tyk2/"
    "industry_benchmarks_network.json"
)
urllib.request.urlretrieve(NETWORK_URL, "industry_benchmarks_network.json")

('industry_benchmarks_network.json', <http.client.HTTPMessage at 0x1063cb250>)

In [2]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"  # colab hack; not needed locally

# Fetch and extract the CLI tutorial results from Zenodo
!openfe fetch rbfe-tutorial-results
!tar -xf rbfe_results.tar.gz

Fetching /Users/hannahbaumann/.local/share/mamba/envs/openfe/lib/python3.13/site-packages/openfecli/tests/data/rbfe_results.tar.gz


## 2. Imports

In [3]:
import json
import pathlib
import warnings

import networkx as nx
from rdkit import Chem

from openfe import (
    AlchemicalNetwork, ChemicalSystem, LigandAtomMapping, LigandNetwork,
    ProteinComponent, SmallMoleculeComponent, SolventComponent, Transformation,
)
from openfe.protocols.openmm_rfe import RelativeHybridTopologyProtocol
from openfe.setup import KartografAtomMapper, lomap_scorers
from openfe.setup.ligand_network_planning import generate_network_from_names
from gufe.tokenization import JSON_HANDLER

from konnektor.network_tools import merge_two_networks
from konnektor.network_planners import MstConcatenator

## 3. Load the planned network

This is the connected network the campaign set out to run.

In [4]:
planned = LigandNetwork.from_json("industry_benchmarks_network.json")

print("connected:", planned.is_connected())
print("ligands:", len(planned.nodes), " edges:", len(planned.edges))

connected: True
ligands: 16  edges: 22


## 4. Parse the results

We read every result JSON, keep only the transformations that succeeded, and
record which ligand pairs finished. An edge counts as **completed** only if it
succeeded in *both* the solvent and complex legs.

Ligand names needed to be adapted in this protocol (a leading `lig_` is stripped) so the tutorial
results line up with the `ejm_*`/`jmc_*` naming in the planned network. We also
store one complex `ChemicalSystem` as a reference for the protein, solvent and
cofactors, as well as a settings object for when we rebuild transformations later.

In [6]:
def _normalise(name: str) -> str:
    return name[4:] if name.startswith("lig_") else name


def _load_result_json(path):
    """Load a result JSON, or None if it isn't a usable result."""
    ru = json.load(open(path, "rb"), cls=JSON_HANDLER.decoder)
    if not isinstance(ru, dict):
        return None  # e.g. the ligand network file
    if ru.get("__qualname__") in ("AlchemicalNetwork", "Transformation"):
        return None  # an input/alchemical network file
    if "unit_results" not in ru:
        return None
    if all("exception" in u for u in ru["unit_results"].values()):
        return None  # every repeat failed
    return ru


def _extract_edge(ru):
    """Return (pair, phase, stateA, settings) for a result, or None if inputs are stripped."""
    data = ru["protocol_result"]["data"]
    units = data[next(iter(data))]
    if not units or "stateA" not in units[0]["inputs"]:
        return None
    inputs = units[0]["inputs"]
    stateA = ChemicalSystem.from_dict(inputs["stateA"])
    mapping = LigandAtomMapping.from_dict(inputs["ligandmapping"])
    pair = frozenset({_normalise(mapping.componentA.name),
                      _normalise(mapping.componentB.name)})
    is_complex = any(isinstance(c, ProteinComponent)
                     for c in stateA.components.values())
    return pair, ("complex" if is_complex else "solvent"), stateA, inputs.get("settings")


def parse_results(result_files, require_both_legs=True):
    """Completed pairs, a reference complex system, and the settings."""
    phases = {}
    reference_complex_system = None
    reference_protocol = None
    for path in result_files:
        ru = _load_result_json(path)
        if ru is None:
            continue
        parsed = _extract_edge(ru)
        if parsed is None:
            continue
        pair, phase, stateA, settings = parsed
        if pair not in phases:
            phases[pair] = set()
        phases[pair].add(phase)
        if phase == "complex" and reference_complex_system is None:
            reference_complex_system = stateA
            if settings is not None:
                reference_protocol = RelativeHybridTopologyProtocol(settings=settings)

    completed_pairs = set()
    for pair, seen in phases.items():
        if not require_both_legs or seen == {"complex", "solvent"}:
            completed_pairs.add(pair)
    return completed_pairs, reference_complex_system, reference_protocol

In [7]:
RESULTS_DIR = pathlib.Path("results")

result_files = sorted(RESULTS_DIR.glob("*/*.json"))
print(f"found {len(result_files)} result files")

completed_pairs, reference_complex_system, reference_protocol = parse_results(result_files)
print("completed edges:", len(completed_pairs))
print("recovered protocol:", reference_protocol is not None)

found 54 result files


/Users/hannahbaumann/.local/share/mamba/envs/openfe/lib/python3.13/site-packages/gufe/components/smallmoleculecomponent.py:286: UserWarning: The atom hybridization data was not found and has been set to unspecified. This can be fixed by recreating the SmallMoleculeComponent from the rdkit molecule after running sanitization.
  warnings.warn(


completed edges: 9
recovered protocol: True


## 5. Reconstruct the completed network and find the breaks

We take the completed pairs and pull the matching edges out of the planned
network — so every ligand keeps the planned network's components. The result is
the network as it actually stands after the campaign, which is where we see the
damage.

In [8]:
def induced_completed(planned, completed_pairs):
    """The completed network as the subnetwork of `planned` that finished."""
    planned_pairs = {
        frozenset({_normalise(e.componentA.name), _normalise(e.componentB.name)}): e
        for e in planned.edges
    }
    edges = [planned_pairs[p] for p in completed_pairs if p in planned_pairs]

    unmatched = completed_pairs - set(planned_pairs)
    if unmatched:
        pairs = ", ".join(" -- ".join(sorted(p)) for p in unmatched)
        warnings.warn(
            f"{len(unmatched)} completed edge(s) are not in the planned network "
            f"and will be ignored: {pairs}. This usually means the results and the "
            f"planned network don't match. Check you supplied the right network."
        )
    return LigandNetwork(edges=edges)


def missing_ligands(planned, completed):
    return set(planned.nodes) - set(completed.nodes)


def decompose_network(network):
    """Split a network into connected pieces."""
    g = network.graph.to_undirected()
    pieces = []
    for comp in nx.connected_components(g):
        edges = [e for e in network.edges
                 if e.componentA in comp and e.componentB in comp]
        pieces.append(LigandNetwork(nodes=comp, edges=edges))
    return pieces


def fragments_to_repair(planned, completed):
    """Connected pieces of `completed`, plus one singleton per missing ligand."""
    frags = decompose_network(completed)
    for lig in missing_ligands(planned, completed):
        frags.append(LigandNetwork(nodes=[lig], edges=[]))
    return frags

In [9]:
completed = induced_completed(planned, completed_pairs)
print("completed connected:", completed.is_connected())
print("missing ligands:", sorted(l.name for l in missing_ligands(planned, completed)))

fragments = fragments_to_repair(planned, completed)
for i, frag in enumerate(fragments):
    print(f"fragment {i}: {sorted(n.name for n in frag.nodes)}")

completed connected: True
missing ligands: ['ejm_44', 'ejm_45', 'ejm_49', 'ejm_50', 'ejm_54', 'ejm_55', 'jmc_23', 'jmc_28', 'jmc_30']
fragment 0: ['ejm_31', 'ejm_42', 'ejm_43', 'ejm_46', 'ejm_47', 'ejm_48', 'jmc_27']
fragment 1: ['jmc_23']
fragment 2: ['ejm_45']
fragment 3: ['ejm_49']
fragment 4: ['ejm_44']
fragment 5: ['ejm_54']
fragment 6: ['jmc_30']
fragment 7: ['ejm_50']
fragment 8: ['ejm_55']
fragment 9: ['jmc_28']


/var/folders/56/3yvgkyg96rvchgjgzfbf16gc0000gn/T/ipykernel_13875/459931657.py:12: UserWarning: 3 completed edge(s) are not in the planned network and will be ignored: ejm_31 -- ejm_50, ejm_46 -- jmc_28, ejm_46 -- jmc_23. This usually means the results and the planned network don't match. Check you supplied the right network.
  warnings.warn(


You'll see a warning that a few completed edges aren't part of the planned network and are being ignored. That's expected here and nothing to worry about: these tutorial results predate the `industry_benchmarks_network` and were run on a different network over the same TYK2 ligands, so a handful of the edges that completed don't exist in the plan we're repairing against. The guard drops them and carries on with the edges the two do share. If this warning appears in your own runs, it means the results and the planned network don't match (most often the wrong planned network was supplied), so it's worth checking.

## 6. Repair — by ligand name

When you know which ligands should bridge the gaps, name the edges explicitly.
Here we chain one representative ligand from each fragment; in practice you'd
choose these from the chemistry.

In [11]:
def repair_by_names(completed, ligands, mapper, names):
    patch = generate_network_from_names(ligands=ligands, mapper=mapper, names=names)
    return merge_two_networks(completed, patch)


reps = [sorted(frag.nodes, key=lambda n: n.name)[0].name for frag in fragments]
bridge_names = list(zip(reps[:-1], reps[1:]))
print("bridging by name:", bridge_names)

mapper = KartografAtomMapper()

repaired_network_by_name = repair_by_names(completed, list(planned.nodes), mapper, bridge_names)
print("connected:", repaired_network_by_name.is_connected())

bridging by name: [('ejm_31', 'jmc_23'), ('jmc_23', 'ejm_45'), ('ejm_45', 'ejm_49'), ('ejm_49', 'ejm_44'), ('ejm_44', 'ejm_54'), ('ejm_54', 'jmc_30'), ('jmc_30', 'ejm_50'), ('ejm_50', 'ejm_55'), ('ejm_55', 'jmc_28')]
connected: True


## 7. Repair — by lomap score

You can also let the scorer choose the bridges: the concatenator proposes candidate edges
between fragments, scores them with lomap, and keeps the best. 
`n_connecting_edges` determines how many edges to connect the fragments with.

In [13]:
def repair_by_score(fragments, mapper, scorer, n_connecting_edges=1):
    concatenator = MstConcatenator(
        mappers=mapper, scorer=scorer, n_connecting_edges=n_connecting_edges
    )
    return concatenator.concatenate_networks(ligand_networks=fragments)


def repair_edges(repaired, completed):
    """The newly added bridges, i.e. what still needs simulating."""
    return list(set(repaired.edges) - set(completed.edges))

mapper = KartografAtomMapper()
scorer = lomap_scorers.default_lomap_score

repaired_network_by_score = repair_by_score(fragments, mapper, scorer, n_connecting_edges=1)
print("connected:", repaired_network_by_score.is_connected())

new_edges = repair_edges(repaired_network_by_score, completed)
print("edges added:", [(e.componentA.name, e.componentB.name) for e in new_edges])

connected: True
edges added: [('ejm_49', 'ejm_44'), ('jmc_23', 'ejm_50'), ('ejm_54', 'ejm_55'), ('jmc_23', 'jmc_28'), ('ejm_49', 'ejm_50'), ('jmc_23', 'jmc_30'), ('ejm_42', 'ejm_54'), ('ejm_45', 'ejm_55'), ('ejm_44', 'jmc_30'), ('jmc_30', 'ejm_55'), ('ejm_31', 'ejm_45'), ('ejm_46', 'ejm_49'), ('jmc_30', 'jmc_28'), ('ejm_49', 'jmc_28'), ('ejm_44', 'ejm_50'), ('ejm_44', 'ejm_54'), ('ejm_54', 'ejm_50'), ('ejm_55', 'jmc_28'), ('ejm_46', 'jmc_30'), ('ejm_45', 'ejm_44'), ('ejm_45', 'jmc_28'), ('ejm_44', 'ejm_55'), ('jmc_27', 'jmc_23'), ('jmc_30', 'ejm_50'), ('ejm_44', 'jmc_28'), ('ejm_49', 'ejm_55'), ('jmc_27', 'jmc_28'), ('jmc_23', 'ejm_45'), ('jmc_23', 'ejm_49'), ('ejm_54', 'jmc_30'), ('ejm_49', 'jmc_30'), ('ejm_50', 'jmc_28'), ('ejm_45', 'ejm_54'), ('jmc_23', 'ejm_44'), ('ejm_42', 'ejm_50'), ('ejm_45', 'jmc_30'), ('jmc_23', 'ejm_54'), ('ejm_50', 'ejm_55'), ('ejm_49', 'ejm_54'), ('ejm_42', 'ejm_55'), ('ejm_43', 'ejm_44'), ('ejm_45', 'ejm_49'), ('jmc_23', 'ejm_55'), ('ejm_45', 'ejm_50'), ('

## 8. Rebuild the alchemical transformations

Turn the new repair edges into runnable transformations — a solvent and a
complex leg each — and write them out. Protein, solvent and cofactors come from
the reference complex system, and the **protocol (with all its settings) is the
one recovered from the results**, so the repair edges match the original run.

The only exception is a charge-changing edge: the completed run may contain no
charge-changing example to copy, so we take the recovered settings and add the
explicit charge correction on top.

In [16]:
import copy


def _formal_charge_difference(mapping):
    a = Chem.rdmolops.GetFormalCharge(mapping.componentA.to_rdkit())
    b = Chem.rdmolops.GetFormalCharge(mapping.componentB.to_rdkit())
    return a - b


def _protocol_for(mapping, reference_protocol):
    """Reuse the campaign's protocol; add charge correction for charge changes."""
    if abs(_formal_charge_difference(mapping)) < 1e-3:
        return reference_protocol
    from openff.units import unit
    settings = copy.deepcopy(reference_protocol.settings)
    settings.alchemical_settings.explicit_charge_correction = True
    settings.simulation_settings.production_length = 20 * unit.nanosecond
    settings.simulation_settings.n_replicas = 22
    settings.lambda_settings.lambda_windows = 22
    return RelativeHybridTopologyProtocol(settings=settings)


def _components_from_reference(reference_complex_system):
    protein = next(c for c in reference_complex_system.components.values()
                   if isinstance(c, ProteinComponent))
    solvent = next((c for c in reference_complex_system.components.values()
                    if isinstance(c, SolventComponent)), SolventComponent())
    cofactors = {name: comp
                 for name, comp in reference_complex_system.components.items()
                 if name.startswith("cofactor")}
    return protein, solvent, cofactors


def build_alchemical_network(repair_network, reference_complex_system, reference_protocol):
    protein, solvent, cofactors = _components_from_reference(reference_complex_system)
    transformations = []
    for mapping in repair_network.edges:
        protocol = _protocol_for(mapping, reference_protocol)
        if protocol is not reference_protocol:
            warnings.warn(
                f"charge-changing edge {mapping.componentA.name} -> "
                f"{mapping.componentB.name}; added explicit charge correction"
            )
        for leg in ("solvent", "complex"):
            sysA = {"ligand": mapping.componentA, "solvent": solvent}
            sysB = {"ligand": mapping.componentB, "solvent": solvent}
            if leg == "complex":
                sysA["protein"] = sysB["protein"] = protein
                sysA.update(cofactors)
                sysB.update(cofactors)
            transformations.append(Transformation(
                stateA=ChemicalSystem(sysA),
                stateB=ChemicalSystem(sysB),
                mapping=mapping,
                protocol=protocol,
                name=f"{leg}_{mapping.componentA.name}_{mapping.componentB.name}",
            ))
    return AlchemicalNetwork(transformations)


def save_transformations(network, out_dir):
    out_dir = pathlib.Path(out_dir)
    (out_dir / "transformations").mkdir(parents=True, exist_ok=True)
    with open(out_dir / "alchemical_network.json", "w") as f:
        json.dump(network.to_dict(), f, cls=JSON_HANDLER.encoder)
    for transform in network.edges:
        transform.to_json(out_dir / "transformations" / f"{transform.name}.json")

In [18]:
# use the repair produced by the score-based strategy
repair_network = LigandNetwork(edges=new_edges)

alchemical_network = build_alchemical_network(
    repair_network, reference_complex_system, reference_protocol
)
save_transformations(alchemical_network, "repair_transformations")
print("wrote", len(alchemical_network.edges), "transformations")

wrote 90 transformations
